In [1]:
import numpy as np
from scipy.linalg import solve_banded
import matplotlib.pyplot as plt

# Параметры задачи
a = 3.0             # скорость волны
h = 0.15            # шаг по пространству
tau = 0.02          # шаг по времени (выбран устойчиво)
x_max = 1.0
t_max = 1.0
d = 0.15            # координата вершины треугольника
l = 0.55            # правая граница треугольника

# Сетка
x = np.arange(0, x_max + h, h)
t = np.arange(0, t_max + tau, tau)
N = len(x) - 1
M = len(t) - 1

# Коэффициент
r = (a * tau / h) ** 2
alpha = a**2 * tau**2 / (2 * h**2)

# Начальное отклонение f(x)
def f(x):
    return 3.0 * np.where(x < d, x / d, np.where(x <= l, (l - x) / (l - d), 0.0))

# Инициализация сетки решения
u = np.zeros((M + 1, N + 1))
u[0, :] = f(x)

# Вычисление второго слоя (при g(x) = 0)
u[1, 1:N] = u[0, 1:N] + (r / 2) * (u[0, 2:N+1] - 2 * u[0, 1:N] + u[0, 0:N-1])

# Формируем трёхдиагональную матрицу A для метода Кранка — Николсона
main_diag = (1 + 2 * alpha) * np.ones(N - 1)
off_diag = -alpha * np.ones(N - 2)

ab = np.zeros((3, N - 1))
ab[0, 1:] = off_diag      # верхняя диагональ
ab[1, :] = main_diag      # главная диагональ
ab[2, :-1] = off_diag     # нижняя диагональ

# Расчёт по времени методом Кранка — Николсона
for j in range(1, M):
    rhs = (2 - 2 * alpha) * u[j, 1:N] + alpha * (u[j, 2:N+1] + u[j, 0:N-1]) - u[j-1, 1:N]
    u[j+1, 1:N] = solve_banded((1, 1), ab, rhs)

# Вывод результатов
for j in range(6):  # первые 6 слоёв
    print(f"t = {t[j]:.2f}: ", ["{:.3f}".format(val) for val in u[j]])


t = 0.00:  ['0.000', '3.000', '1.875', '0.750', '0.000', '0.000', '0.000', '0.000']
t = 0.02:  ['0.000', '2.670', '1.875', '0.780', '0.060', '0.000', '0.000', '0.000']
t = 0.04:  ['0.000', '1.901', '1.786', '0.862', '0.210', '0.019', '0.001', '0.000']
t = 0.06:  ['0.000', '0.944', '1.539', '0.967', '0.413', '0.073', '0.008', '0.000']
t = 0.08:  ['0.000', '0.040', '1.108', '1.045', '0.631', '0.175', '0.029', '0.000']
t = 0.10:  ['0.000', '-0.636', '0.534', '1.038', '0.822', '0.321', '0.073', '0.000']
